# Migration: Increment_data
**Source File:** target_sql_file.sql  
**Target Workspace:** Databricks Delta Lake  
**Conversion Date:** 2023-10-27

In [ ]:
dbutils.widgets.text("ODI_SESS_NO", "4305d7be-9b95-4927-a037-43faaed2041c")
dbutils.widgets.text("v_lst_dt", "01-01-00")

## ETL Parameter Setup

In [ ]:
%sql
-- SCEN_TASK_NO {1}: Get Last Run Date
CREATE OR REPLACE TEMPORARY VIEW v_last_run_params AS
SELECT 
    COALESCE(
        (SELECT date_format(last_run_dt, 'dd-MM-yy') FROM workspace.odi_trg.log_table1 WHERE pkg_name = 'Increment_data'), 
        date_format(current_timestamp(), 'dd-MM-yy')
    ) AS last_run_dt_str;

-- SCEN_TASK_NO {2}: Update Log Table
UPDATE workspace.odi_trg.log_table1 
SET last_run_dt = current_timestamp() 
WHERE pkg_name = 'Increment_data';

## Staging Table (C$)

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Drop Staging Table
DROP TABLE IF EXISTS workspace.odi_trg.c_employees_stg;

In [ ]:
%sql
-- SCEN_TASK_NO {40}: Create Staging Table
CREATE TABLE workspace.odi_trg.c_employees_stg
(
    EMPLOYEE_ID    BIGINT,
    FIRST_NAME     STRING,
    LAST_NAME      STRING,
    EMAIL          STRING,
    PHONE_NUMBER   STRING,
    HIRE_DATE      TIMESTAMP,
    JOB_ID         STRING,
    SALARY         DECIMAL(8,2),
    COMMISSION_PCT DECIMAL(2,2),
    MANAGER_ID     BIGINT,
    DEPARTMENT_ID  BIGINT,
    LAST_UPD_DT    TIMESTAMP
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {50}: Extract Source Data into Staging
INSERT INTO workspace.odi_trg.c_employees_stg
(
    EMPLOYEE_ID,
    FIRST_NAME,
    LAST_NAME,
    EMAIL,
    PHONE_NUMBER,
    HIRE_DATE,
    JOB_ID,
    SALARY,
    COMMISSION_PCT,
    MANAGER_ID,
    DEPARTMENT_ID,
    LAST_UPD_DT
)
SELECT
    SRC_EMPLOYEES_1.EMPLOYEE_ID,
    SRC_EMPLOYEES_1.FIRST_NAME,
    SRC_EMPLOYEES_1.LAST_NAME,
    SRC_EMPLOYEES_1.EMAIL,
    SRC_EMPLOYEES_1.PHONE_NUMBER,
    SRC_EMPLOYEES_1.HIRE_DATE,
    SRC_EMPLOYEES_1.JOB_ID,
    SRC_EMPLOYEES_1.SALARY,
    SRC_EMPLOYEES_1.COMMISSION_PCT,
    SRC_EMPLOYEES_1.MANAGER_ID,
    SRC_EMPLOYEES_1.DEPARTMENT_ID,
    SRC_EMPLOYEES_1.LAST_UPD_DT
FROM workspace.odi_src.src_employees AS SRC_EMPLOYEES_1
WHERE (SRC_EMPLOYEES_1.LAST_UPD_DT >= to_date('${v_lst_dt}', 'dd-MM-yy'));

## Flow Table (I$)

In [ ]:
%sql
-- SCEN_TASK_NO {70}: Drop Flow Table
DROP TABLE IF EXISTS workspace.odi_trg.i_employees_flow;

In [ ]:
%sql
-- SCEN_TASK_NO {80}: Create Flow Table
CREATE TABLE workspace.odi_trg.i_employees_flow
(
    EMPLOYEE_ID    BIGINT,
    FIRST_NAME     STRING,
    LAST_NAME      STRING,
    EMAIL          STRING,
    PHONE_NUMBER   STRING,
    HIRE_DATE      TIMESTAMP,
    JOB_ID         STRING,
    SALARY         DECIMAL(8,2),
    COMMISSION_PCT DECIMAL(2,2),
    MANAGER_ID     BIGINT,
    DEPARTMENT_ID  BIGINT,
    LAST_UPD_DT    TIMESTAMP,
    IND_UPDATE     STRING
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {90}: Insert New Records into Flow
INSERT INTO workspace.odi_trg.i_employees_flow
(
    EMPLOYEE_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE_NUMBER, HIRE_DATE, JOB_ID, SALARY, COMMISSION_PCT, MANAGER_ID, DEPARTMENT_ID, LAST_UPD_DT, IND_UPDATE
)
SELECT 
    S.EMPLOYEE_ID, S.FIRST_NAME, S.LAST_NAME, S.EMAIL, S.PHONE_NUMBER, S.HIRE_DATE, S.JOB_ID, S.SALARY, S.COMMISSION_PCT, S.MANAGER_ID, S.DEPARTMENT_ID, S.LAST_UPD_DT, S.IND_UPDATE
FROM (
    SELECT 
        FILTER_A.EMPLOYEE_ID, FILTER_A.FIRST_NAME, FILTER_A.LAST_NAME, FILTER_A.EMAIL, FILTER_A.PHONE_NUMBER, FILTER_A.HIRE_DATE, FILTER_A.JOB_ID, FILTER_A.SALARY, FILTER_A.COMMISSION_PCT, FILTER_A.MANAGER_ID, FILTER_A.DEPARTMENT_ID, FILTER_A.LAST_UPD_DT,
        'I' AS IND_UPDATE
    FROM workspace.odi_trg.c_employees_stg AS FILTER_A
) AS S
WHERE NOT EXISTS (
    SELECT 1 FROM workspace.odi_trg.trg_employees AS T
    WHERE T.EMPLOYEE_ID = S.EMPLOYEE_ID 
    AND (T.FIRST_NAME = S.FIRST_NAME OR (T.FIRST_NAME IS NULL AND S.FIRST_NAME IS NULL))
    AND (T.LAST_NAME = S.LAST_NAME OR (T.LAST_NAME IS NULL AND S.LAST_NAME IS NULL))
    AND (T.EMAIL = S.EMAIL OR (T.EMAIL IS NULL AND S.EMAIL IS NULL))
    AND (T.PHONE_NUMBER = S.PHONE_NUMBER OR (T.PHONE_NUMBER IS NULL AND S.PHONE_NUMBER IS NULL))
    AND (T.HIRE_DATE = S.HIRE_DATE OR (T.HIRE_DATE IS NULL AND S.HIRE_DATE IS NULL))
    AND (T.JOB_ID = S.JOB_ID OR (T.JOB_ID IS NULL AND S.JOB_ID IS NULL))
    AND (T.SALARY = S.SALARY OR (T.SALARY IS NULL AND S.SALARY IS NULL))
    AND (T.COMMISSION_PCT = S.COMMISSION_PCT OR (T.COMMISSION_PCT IS NULL AND S.COMMISSION_PCT IS NULL))
    AND (T.MANAGER_ID = S.MANAGER_ID OR (T.MANAGER_ID IS NULL AND S.MANAGER_ID IS NULL))
    AND (T.DEPARTMENT_ID = S.DEPARTMENT_ID OR (T.DEPARTMENT_ID IS NULL AND S.DEPARTMENT_ID IS NULL))
    AND (T.LAST_UPD_DT = S.LAST_UPD_DT OR (T.LAST_UPD_DT IS NULL AND S.LAST_UPD_DT IS NULL))
);

## Data Quality and Error Handling

In [ ]:
%sql
-- SCEN_TASK_NO {100-110}: Optimize Flow Table
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.odi_trg.i_employees_flow ZORDER BY (EMPLOYEE_ID);

In [ ]:
%sql
-- SCEN_TASK_NO {120}: Create SNP_CHECK_TAB
CREATE TABLE IF NOT EXISTS workspace.odi_trg.snp_check_tab
(
    CATALOG_NAME    STRING,
    SCHEMA_NAME     STRING,
    RESOURCE_NAME   STRING,
    FULL_RES_NAME   STRING,
    ERR_TYPE        STRING,
    ERR_MESS        STRING,
    CHECK_DATE      TIMESTAMP,
    ORIGIN          STRING,
    CONS_NAME       STRING,
    CONS_TYPE       STRING,
    ERR_COUNT       BIGINT
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {130}: Create E$ Error Table
CREATE TABLE IF NOT EXISTS workspace.odi_trg.e_trg_employees
(
    ODI_ROW_ID      STRING,
    ODI_ERR_TYPE    STRING, 
    ODI_ERR_MESS    STRING,
    ODI_CHECK_DATE  TIMESTAMP, 
    EMPLOYEE_ID     BIGINT,
    FIRST_NAME      STRING,
    LAST_NAME       STRING,
    EMAIL           STRING,
    PHONE_NUMBER    STRING,
    HIRE_DATE       TIMESTAMP,
    JOB_ID          STRING,
    SALARY          DECIMAL(8,2),
    COMMISSION_PCT  DECIMAL(2,2),
    MANAGER_ID      BIGINT,
    DEPARTMENT_ID   BIGINT,
    LAST_UPD_DT     TIMESTAMP,
    ODI_ORIGIN      STRING,
    ODI_CONS_NAME   STRING,
    ODI_CONS_TYPE   STRING,
    ODI_PK          STRING,
    ODI_SESS_NO     STRING
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {140}: Cleanup Previous Errors for Session
DELETE FROM workspace.odi_trg.e_trg_employees
WHERE ODI_ORIGIN = '(4)test.Increment';

In [ ]:
%sql
-- SCEN_TASK_NO {150-180}: Validate Not Null Constraints
INSERT INTO workspace.odi_trg.e_trg_employees
(
    ODI_PK, ODI_SESS_NO, ODI_ROW_ID, ODI_ERR_TYPE, ODI_ERR_MESS, ODI_CHECK_DATE, ODI_ORIGIN, ODI_CONS_NAME, ODI_CONS_TYPE, EMPLOYEE_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE_NUMBER, HIRE_DATE, JOB_ID, SALARY, COMMISSION_PCT, MANAGER_ID, DEPARTMENT_ID, LAST_UPD_DT
)
SELECT
    uuid(), '${ODI_SESS_NO}', CAST(monotonically_increasing_id() AS STRING), 'F', 'ODI-15066: The column LAST_NAME cannot be null.', current_timestamp(), '(4)test.Increment', 'LAST_NAME', 'NN', EMPLOYEE_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE_NUMBER, HIRE_DATE, JOB_ID, SALARY, COMMISSION_PCT, MANAGER_ID, DEPARTMENT_ID, LAST_UPD_DT
FROM workspace.odi_trg.i_employees_flow
WHERE LAST_NAME IS NULL;

INSERT INTO workspace.odi_trg.e_trg_employees
(
    ODI_PK, ODI_SESS_NO, ODI_ROW_ID, ODI_ERR_TYPE, ODI_ERR_MESS, ODI_CHECK_DATE, ODI_ORIGIN, ODI_CONS_NAME, ODI_CONS_TYPE, EMPLOYEE_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE_NUMBER, HIRE_DATE, JOB_ID, SALARY, COMMISSION_PCT, MANAGER_ID, DEPARTMENT_ID, LAST_UPD_DT
)
SELECT
    uuid(), '${ODI_SESS_NO}', CAST(monotonically_increasing_id() AS STRING), 'F', 'ODI-15066: The column EMAIL cannot be null.', current_timestamp(), '(4)test.Increment', 'EMAIL', 'NN', EMPLOYEE_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE_NUMBER, HIRE_DATE, JOB_ID, SALARY, COMMISSION_PCT, MANAGER_ID, DEPARTMENT_ID, LAST_UPD_DT
FROM workspace.odi_trg.i_employees_flow
WHERE EMAIL IS NULL;

INSERT INTO workspace.odi_trg.e_trg_employees
(
    ODI_PK, ODI_SESS_NO, ODI_ROW_ID, ODI_ERR_TYPE, ODI_ERR_MESS, ODI_CHECK_DATE, ODI_ORIGIN, ODI_CONS_NAME, ODI_CONS_TYPE, EMPLOYEE_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE_NUMBER, HIRE_DATE, JOB_ID, SALARY, COMMISSION_PCT, MANAGER_ID, DEPARTMENT_ID, LAST_UPD_DT
)
SELECT
    uuid(), '${ODI_SESS_NO}', CAST(monotonically_increasing_id() AS STRING), 'F', 'ODI-15066: The column HIRE_DATE cannot be null.', current_timestamp(), '(4)test.Increment', 'HIRE_DATE', 'NN', EMPLOYEE_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE_NUMBER, HIRE_DATE, JOB_ID, SALARY, COMMISSION_PCT, MANAGER_ID, DEPARTMENT_ID, LAST_UPD_DT
FROM workspace.odi_trg.i_employees_flow
WHERE HIRE_DATE IS NULL;

INSERT INTO workspace.odi_trg.e_trg_employees
(
    ODI_PK, ODI_SESS_NO, ODI_ROW_ID, ODI_ERR_TYPE, ODI_ERR_MESS, ODI_CHECK_DATE, ODI_ORIGIN, ODI_CONS_NAME, ODI_CONS_TYPE, EMPLOYEE_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE_NUMBER, HIRE_DATE, JOB_ID, SALARY, COMMISSION_PCT, MANAGER_ID, DEPARTMENT_ID, LAST_UPD_DT
)
SELECT
    uuid(), '${ODI_SESS_NO}', CAST(monotonically_increasing_id() AS STRING), 'F', 'ODI-15066: The column JOB_ID cannot be null.', current_timestamp(), '(4)test.Increment', 'JOB_ID', 'NN', EMPLOYEE_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE_NUMBER, HIRE_DATE, JOB_ID, SALARY, COMMISSION_PCT, MANAGER_ID, DEPARTMENT_ID, LAST_UPD_DT
FROM workspace.odi_trg.i_employees_flow
WHERE JOB_ID IS NULL;

In [ ]:
%sql
-- SCEN_TASK_NO {200}: Remove Erroneous Records from Flow
MERGE INTO workspace.odi_trg.i_employees_flow AS T
USING (
    SELECT DISTINCT EMPLOYEE_ID
    FROM workspace.odi_trg.e_trg_employees
    WHERE ODI_SESS_NO = '${ODI_SESS_NO}'
) AS S
ON T.EMPLOYEE_ID = S.EMPLOYEE_ID
WHEN MATCHED THEN DELETE;

In [ ]:
%sql
-- SCEN_TASK_NO {210}: Log Quality Issues Summary
INSERT INTO workspace.odi_trg.snp_check_tab
(
    SCHEMA_NAME, RESOURCE_NAME, FULL_RES_NAME, ERR_TYPE, ERR_MESS, CHECK_DATE, ORIGIN, CONS_NAME, CONS_TYPE, ERR_COUNT
)
SELECT 
    'ODI_TRG', 'TRG_EMPLOYEES', 'workspace.odi_trg.trg_employees', E.ODI_ERR_TYPE, E.ODI_ERR_MESS, E.ODI_CHECK_DATE, E.ODI_ORIGIN, E.ODI_CONS_NAME, E.ODI_CONS_TYPE, COUNT(1) 
FROM workspace.odi_trg.e_trg_employees AS E
WHERE E.ODI_ERR_TYPE = 'F' AND E.ODI_ORIGIN = '(4)test.Increment'
GROUP BY E.ODI_ERR_TYPE, E.ODI_ERR_MESS, E.ODI_CHECK_DATE, E.ODI_ORIGIN, E.ODI_CONS_NAME, E.ODI_CONS_TYPE;

## Mark Records for Update

In [ ]:
%sql
-- SCEN_TASK_NO {220}: Flag Existing Records for Update
MERGE INTO workspace.odi_trg.i_employees_flow AS T
USING (
    SELECT EMPLOYEE_ID FROM workspace.odi_trg.trg_employees
) AS S
ON T.EMPLOYEE_ID = S.EMPLOYEE_ID
WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';

## Final Merge into Target

In [ ]:
%sql
-- SCEN_TASK_NO {260+270}: Merge Changes into Target Table
MERGE INTO workspace.odi_trg.trg_employees AS T
USING workspace.odi_trg.i_employees_flow AS S
ON T.EMPLOYEE_ID = S.EMPLOYEE_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.FIRST_NAME     = S.FIRST_NAME,
    T.LAST_NAME      = S.LAST_NAME,
    T.EMAIL          = S.EMAIL,
    T.PHONE_NUMBER   = S.PHONE_NUMBER,
    T.HIRE_DATE      = S.HIRE_DATE,
    T.JOB_ID         = S.JOB_ID,
    T.SALARY         = S.SALARY,
    T.COMMISSION_PCT = S.COMMISSION_PCT,
    T.MANAGER_ID     = S.MANAGER_ID,
    T.DEPARTMENT_ID  = S.DEPARTMENT_ID,
    T.LAST_UPD_DT    = S.LAST_UPD_DT
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT
(
    EMPLOYEE_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE_NUMBER, HIRE_DATE, JOB_ID, SALARY, COMMISSION_PCT, MANAGER_ID, DEPARTMENT_ID, LAST_UPD_DT
)
VALUES
(
    S.EMPLOYEE_ID, S.FIRST_NAME, S.LAST_NAME, S.EMAIL, S.PHONE_NUMBER, S.HIRE_DATE, S.JOB_ID, S.SALARY, S.COMMISSION_PCT, S.MANAGER_ID, S.DEPARTMENT_ID, S.LAST_UPD_DT
);

## Post-Load Optimization

In [ ]:
%sql
-- Disable ZORDER stats check
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.odi_trg.trg_employees ZORDER BY (EMPLOYEE_ID);

## Cleanup

In [ ]:
%sql
-- SCEN_TASK_NO {330, 350}: Drop Working Tables
DROP TABLE IF EXISTS workspace.odi_trg.c_employees_stg;
DROP TABLE IF EXISTS workspace.odi_trg.i_employees_flow;